# SeeDL · Neural network explorer

Real neurons, real weights, real activations. This executed notebook demonstrates the visual
redesign on deterministic, randomly initialized toy models (these are not trained models).

## Setup
Install `seedl[all]` and a Jupyter kernel. Run every cell in order. No data is downloaded.
Each example exports a self-contained interactive HTML file to `docs/visuals`.
The 3D views require a WebGL-capable browser and trusted notebook output.


In [1]:
from pathlib import Path

import plotly.io as pio
import torch
from torch import nn

import seedl as sd

pio.renderers.default = "plotly_mimetype"
torch.manual_seed(7)
root = Path.cwd()
if root.name == "examples":
    root = root.parent
out = root / "docs" / "visuals"
out.mkdir(parents=True, exist_ok=True)
model = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 6), nn.ReLU(), nn.Linear(6, 2))
x = torch.tensor([[0.3, 0.8, -0.2, 1.0], [-0.5, 0.1, 0.7, 0.2]])
print("SeeDL", sd.__version__, "| PyTorch", torch.__version__)

SeeDL 0.1.0 | PyTorch 2.14.1+cu130


## 1. Recognize the network
Four input neurons, eight and six hidden neurons, two output neurons. ReLU belongs to its
transformation, so there are **20 nodes and 92 weighted connections**, not six neuron layers.
Colors identify layers when no input is supplied. Hover on a connection to read its real weight.


In [2]:
small = sd.see(model)
assert len(small.data["neuron_graph"]["nodes"]) == 20
assert len(small.data["neuron_graph"]["edges"]) == 92
small.save(out / "small-light.html")
small.save(out / "small-light.svg")
small

Neural network: 6 module(s) · 108 parameter(s) · 108 trainable
  Summary: 6 module(s) · 108 parameter(s) · 108 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: Edges use weight[destination, source]. Teal is positive; coral is negative; gray is zero; width encodes magnitude in four bins.
  Note: Node activation is the captured post-activation output; bias and pre-activation appear on hover. No backward pass is run.
  Note: No activations captured. Pass input=... to observe a forward pass.

## 2. A medium MLP with bounded connections
This 80-neuron model has 1,536 connections. The preview budget samples at most 250 edges,
evenly across matrix indices. The disclosure stays in the chart; unshown edges still exist.


In [3]:
medium_model = nn.Sequential(nn.Linear(32, 32), nn.Tanh(), nn.Linear(32, 16))
medium = sd.see(medium_model, max_connections=250)
medium.save(out / "medium.html")
medium

Neural network: 4 module(s) · 1,584 parameter(s) · 1,584 trainable
  Summary: 4 module(s) · 1,584 parameter(s) · 1,584 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: Edges use weight[destination, source]. Teal is positive; coral is negative; gray is zero; width encodes magnitude in four bins.
  Note: Node activation is the captured post-activation output; bias and pre-activation appear on hover. No backward pass is run.
  Note: Connections are deterministically sampled before magnitude filtering: showing 220 of 1,536.
  Note: No activations captured. Pass input=... to observe a forward pass.

## 3. Filter by real weight magnitude
Teal connections are positive and coral connections are negative. Width uses four magnitude
bins within the displayed weights. Filtering makes a clearer picture, but hides weak connections.


In [4]:
weighted = sd.see(model, min_weight=0.25, edge_opacity=0.6)
weighted.save(out / "weighted.html")
weighted

Neural network: 6 module(s) · 108 parameter(s) · 108 trainable
  Summary: 6 module(s) · 108 parameter(s) · 108 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: Edges use weight[destination, source]. Teal is positive; coral is negative; gray is zero; width encodes magnitude in four bins.
  Note: Node activation is the captured post-activation output; bias and pre-activation appear on hover. No backward pass is run.
  Note: Connections with |weight| < 0.25 are hidden after sampling.
  Note: No activations captured. Pass input=... to observe a forward pass.

## 4. Observe one real sample
Node colors now encode actual outputs from sample 0 of the supplied batch. Hidden nodes show
the output **after ReLU**; hover exposes the pre-activation and bias too. Zero is a real value;
missing or non-finite values are not replaced by zero. One eval/no-grad pass uses cloned input.


In [5]:
active = sd.see(model, input=x, sample=0, theme="dark")
expected = model(x)[0].detach().tolist()
actual = [node["activation"] for node in active.data["neuron_graph"]["layers"][-1]["nodes"]]
assert actual == expected
active.save(out / "neurons.html")
active.save(out / "neurons.svg")
active

Neural network: 6 module(s) · 108 parameter(s) · 108 trainable
  Summary: 6 module(s) · 108 parameter(s) · 108 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: An eval-mode forward pass under no_grad() was run once with the supplied input; temporary hooks were removed and ordinary buffer updates were restored.
  Note: Edges use weight[destination, source]. Teal is positive; coral is negative; gray is zero; width encodes magnitude in four bins.
  Note: Node activation is the captured post-activation output; bias and pre-activation appear on hover. No backward pass is run.
  Note: Activations show sample 0 (zero-based), from one eval/no_grad pass on a cloned input.

## 5. Select and inspect an individual neuron
Select any neuron (e.g. `select_neuron=(1, 2)`) to isolate its incoming (cyan) and outgoing (amber)
weighted connections while dimming unrelated edges. The interactive dropdown lets you switch
between overview and any individual neuron, revealing real bias, activation, pre-activation,
and weights in the inspection panel below the network.


In [6]:
inspected = sd.see(model, input=x, select_neuron=(1, 2), theme="dark")
inspected.save(out / "neuron-selected.html")
inspected.save(out / "neuron-selected.svg")
inspected

Neural network: 6 module(s) · 108 parameter(s) · 108 trainable
  Summary: 6 module(s) · 108 parameter(s) · 108 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: An eval-mode forward pass under no_grad() was run once with the supplied input; temporary hooks were removed and ordinary buffer updates were restored.
  Note: Edges use weight[destination, source]. Teal is positive; coral is negative; gray is zero; width encodes magnitude in four bins.
  Note: Node activation is the captured post-activation output; bias and pre-activation appear on hover. No backward pass is run.
  Note: Activations show sample 0 (zero-based), from one eval/no_grad pass on a cloned input.

## 6. Rotate a real 3D network
Neurons occupy deterministic grids within layer planes. Drag to orbit; choose zoom or pan in
the toolbar. **Orbit**, **Pan**, and **Reset view** are also available above the graph.
Dotted outlines identify layer planes; they are not connections. Labels
can overlap at edge-on camera angles. Spatial positions are layout coordinates, not learned values.


In [7]:
spatial = sd.see(model, input=x, view="3d", theme="dark")
assert all(trace.type == "scatter3d" for trace in spatial.figure.data)
spatial.save(out / "3d.html")
spatial.save(out / "3d.svg")
spatial

Neural network · 3D: 6 module(s) · 108 parameter(s) · 108 trainable
  Summary: 6 module(s) · 108 parameter(s) · 108 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: An eval-mode forward pass under no_grad() was run once with the supplied input; temporary hooks were removed and ordinary buffer updates were restored.
  Note: Edges use weight[destination, source]. Teal is positive; coral is negative; gray is zero; width encodes magnitude in four bins.
  Note: Node activation is the captured post-activation output; bias and pre-activation appear on hover. No backward pass is run.
  Note: Activations show sample 0 (zero-based), from one eval/no_grad pass on a cloned input.

### Compare light mode and a denser 3D view
The same camera and layer guides work in both themes. The medium model still uses the edge
budget; all 80 neurons are present while the caption discloses sampled connections.


In [8]:
spatial_light = sd.see(model, input=x, view="3d", theme="light")
spatial_light.save(out / "3d-light.html")
spatial_light

Neural network · 3D: 6 module(s) · 108 parameter(s) · 108 trainable
  Summary: 6 module(s) · 108 parameter(s) · 108 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: An eval-mode forward pass under no_grad() was run once with the supplied input; temporary hooks were removed and ordinary buffer updates were restored.
  Note: Edges use weight[destination, source]. Teal is positive; coral is negative; gray is zero; width encodes magnitude in four bins.
  Note: Node activation is the captured post-activation output; bias and pre-activation appear on hover. No backward pass is run.
  Note: Activations show sample 0 (zero-based), from one eval/no_grad pass on a cloned input.

In [9]:
medium_spatial = sd.see(medium_model, view="3d", max_connections=250)
medium_spatial.save(out / "medium-3d.html")
medium_spatial

Neural network · 3D: 4 module(s) · 1,584 parameter(s) · 1,584 trainable
  Summary: 4 module(s) · 1,584 parameter(s) · 1,584 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: Edges use weight[destination, source]. Teal is positive; coral is negative; gray is zero; width encodes magnitude in four bins.
  Note: Node activation is the captured post-activation output; bias and pre-activation appear on hover. No backward pass is run.
  Note: Connections are deterministically sampled before magnitude filtering: showing 220 of 1,536.
  Note: No activations captured. Pass input=... to observe a forward pass.

## 7. Follow forward propagation with animation
The forward explorer offers interactive stepping and playback (**▶ Play**, **⏸ Pause**, **⏮ Reset**, with **0.5x**, **1x**, **2x** speed controls).
Highlighting represents computational execution order through layers, displaying actual captured
activations from the evaluation pass without fabricating artificial intermediate states.
The slider and playback buttons are above the graph; the information panel updates with actual layer tensors.


In [10]:
forward = sd.see(model, input=x, theme="dark")
forward_figure = forward.explore()
forward.save(out / "forward.html")
forward_3d = sd.see(model, input=x, view="3d", theme="dark")
forward_3d.explore()
forward_3d.save(out / "forward-3d.html")
forward_sampled = sd.see(model, input=x, max_connections=40)
forward_sampled.explore()
forward_sampled.save(out / "forward-sampled.html")
forward_figure

## 8. Convolutional data flow
Convolution blocks show channel counts, kernel/stride and captured spatial dimensions.
These are operations and feature-map shapes, not fully connected neuron wiring.


In [11]:
cnn = nn.Sequential(
    nn.Conv2d(3, 8, 3, padding=1),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d((2, 2)),
    nn.Flatten(),
    nn.Linear(32, 3),
)
conv = sd.see(cnn, input=torch.ones(1, 3, 16, 16))
conv.save(out / "cnn.html")
conv.save(out / "cnn.svg")
conv

Sequential architecture: 6 module(s) · 323 parameter(s) · 323 trainable
  Summary: 6 module(s) · 323 parameter(s) · 323 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: An eval-mode forward pass under no_grad() was run once with the supplied input; temporary hooks were removed and ordinary buffer updates were restored.
  Note: Sequential data flow · blocks represent operations, not individual neurons.

## 9. Large models collapse before extracting weights
A 1,024 → 2,048 → 10 model has over two million weights. It becomes a handful of architecture
blocks. No million-edge graph is constructed. `view="summary"` retains detailed inspection.


In [12]:
large_model = nn.Sequential(nn.Linear(1024, 2048), nn.ReLU(), nn.Linear(2048, 10))
large = sd.see(large_model)
assert "neuron_graph" not in large.data
large.save(out / "large.html")
large

Sequential architecture: 4 module(s) · 2,119,690 parameter(s) · 2,119,690 trainable
  Summary: 4 module(s) · 2,119,690 parameter(s) · 2,119,690 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: Sequential data flow · blocks represent operations, not individual neurons.
  Note: Neuron detail collapsed: 3,082 neurons; limit 100, 48 per layer and 12 layers. Showing architecture blocks.

## 10. Transformer hierarchy
Custom/residual structures use containment connectors. Registration order does not prove
execution order; this view does not invent skip connections. Attention, feed-forward and
normalization modules remain visible, with deeper details in `view="summary"`.


In [13]:
transformer = nn.TransformerEncoderLayer(d_model=16, nhead=4, dim_feedforward=32, batch_first=True)
hierarchy = sd.see(transformer)
hierarchy.save(out / "transformer.html")
hierarchy

TransformerEncoderLayer architecture: 10 module(s) · 2,224 parameter(s) · 2,224 trainable
  Summary: 10 module(s) · 2,224 parameter(s) · 2,224 trainable
  Note: Per-layer counts include direct parameters only; shared parameters may appear in multiple rows. Total counts deduplicate shared parameters.
  Note: Module hierarchy · connectors show containment, not execution or residual paths.

## 11. Three-component PCA
These are explicitly generated example vectors, not real documents. PCA fits the displayed
sample and reports explained variance per axis. Category colors use supplied labels. Coordinates
are scaled by the recorded `coordinate_scale` before PCA to avoid numeric overflow.


In [14]:
vectors = torch.randn(48, 6)
vectors[24:] += torch.tensor([2.0, -1.0, 0.0, 1.0, 0.0, 0.0])
embedded = sd.embeddings(
    vectors,
    dimensions=3,
    labels=[f"Example {i}" for i in range(48)],
    categories=["Group A"] * 24 + ["Group B"] * 24,
)
embedded.save(out / "embeddings-3d.html")
embedded

Embeddings · 3D PCA: PCA fitted to 48 of 48 evenly spaced vectors; coordinates scaled by 4.4895.
  Note: PCA fitted to 48 of 48 evenly spaced vectors; coordinates scaled by 4.4895.
  Note: First three components explain 70.2% of sampled variance. Projection can distort distances.

## 12. A defined loss slice
For a one-input linear regressor, vary weight[0] and bias[0] around the current model. All other
parameters stay fixed. `evaluate=True` explicitly authorizes repeated evaluation on a deep copy.
This is a sampled two-dimensional coordinate slice of parameter space, not the full loss function.
The supplied path is an illustrative slice trajectory, not a recorded optimizer run.


In [15]:
regressor = nn.Linear(1, 1)
with torch.no_grad():
    regressor.weight.fill_(1.0)
    regressor.bias.fill_(0.0)
features = torch.linspace(-1, 1, 20).reshape(-1, 1)
targets = 2 * features + 0.5
surface = sd.loss_landscape(
    regressor,
    (features, targets),
    loss_fn=nn.functional.mse_loss,
    axes=(("weight", 0), ("bias", 0)),
    radius=1.5,
    steps=15,
    evaluate=True,
    trajectory=[(0, 0), (0.5, 0.25), (1, 0.5)],
)
surface.save(out / "loss-slice.html")
surface

Loss landscape · coordinate slice: A sampled two-dimensional coordinate slice, not the full loss landscape: 15 × 15 forward evaluations.
  Note: A sampled two-dimensional coordinate slice, not the full loss landscape: 15 × 15 forward evaluations.
  Note: All other parameters held fixed. Eval/no_grad on a deep copy; no backward pass. CPU RNG restored.
  Note: Trajectory, when supplied, is evaluated in this same slice; it is not an automatically captured optimizer path.

## Checks and limits

- Neuron detail: exact built-in Linear/Sequential plus supported elementwise activations; no custom hooks.
- Default limits: 100 total neurons, 48 per layer, 12 layers, 1,500 sampled connections.
- Interactive inspection: dropdown neuron selection highlights incoming (cyan) and outgoing (amber) connections, and shows exact bias, activation, pre-activation, and weights.
- Forward propagation: animated playback (**▶ Play**, **⏸ Pause**, **⏮ Reset**) and layer stepping operate purely on captured data without fabricating values.
- Higher-rank MLP activations require selecting a `[features]` or `[batch, features]` input first.
- SVG remains usable without Plotly; a 3D neuron SVG is explicitly labeled as a 2D projection.
- Arbitrary directed acyclic computation graphs, complex branching, and automatic optimizer-path capture are not implemented.
- All other SeeDL entry points remain available; this notebook focuses on the visual redesign.
